# 실패를 처리하는 라우팅

분류값을 허용 목록과 신뢰도 기준으로 검사한 뒤 전문 프롬프트로 분기합니다.

## 실행 환경과 비용 상한

최대 4회, 호출당 출력 220토큰으로 제한합니다. 계획 비용은 0.0100달러 미만입니다.

In [1]:
# 이 장은 OpenAI SDK를 직접 배우므로 두 패키지만 고정합니다.
%pip install -q openai==2.29.0 python-dotenv==1.2.3


Note: you may need to restart the kernel to use updated packages.


In [2]:
# 동기 API 호출과 사용량 측정에 필요한 도구를 준비합니다.
import json
import os
import time
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "프로젝트 루트 .env에 OPENAI_API_KEY가 필요합니다."
MODEL = "gpt-5.4-mini"
client = OpenAI(timeout=45.0, max_retries=0)
calls: list[dict[str, float | int | str]] = []
print("모델:", MODEL, "| API 키 존재: 예")


모델: gpt-5.4-mini | API 키 존재: 예


In [3]:
# 동기 Responses API의 JSON 응답과 사용량을 기록합니다.
def call_json(instructions: str, prompt: str, limit: int = 240) -> dict:
    started = time.perf_counter()
    response = client.responses.create(
        model=MODEL,
        instructions=instructions + " 반드시 유효한 JSON 객체만 반환하세요.",
        input="JSON 요청\n" + prompt,
        max_output_tokens=limit,
        reasoning={"effort": "none"},
        text={"format": {"type": "json_object"}},
    )
    elapsed = time.perf_counter() - started
    usage = response.usage
    calls.append({"latency_s": elapsed, "input": usage.input_tokens,
                  "output": usage.output_tokens})
    return json.loads(response.output_text)


In [4]:
# 동기 응답의 answer 형식을 엄격하게 검사합니다.
def call_answer(instructions: str, prompt: str, limit: int = 400) -> dict:
    started = time.perf_counter()
    response = client.responses.create(
        model=MODEL, instructions=instructions, input=prompt,
        max_output_tokens=limit, reasoning={"effort": "none"},
        text={"format": {"type": "json_schema", "name": "answer_payload",
              "schema": {"type": "object", "properties": {"answer": {"type": "string"}},
              "required": ["answer"], "additionalProperties": False}, "strict": True}},
    )
    calls.append({"latency_s": time.perf_counter() - started,
                  "input": response.usage.input_tokens, "output": response.usage.output_tokens})
    data = json.loads(response.output_text)
    if not isinstance(data.get("answer"), str) or not data["answer"].strip():
        raise ValueError("응답의 answer가 비어 있습니다.")
    return data


In [5]:
# 반환된 토큰 사용량으로 할인 없는 비용을 추정합니다.
def cost_usd(records: list[dict]) -> float:
    inputs = sum(int(row["input"]) for row in records)
    outputs = sum(int(row["output"]) for row in records)
    return inputs * 0.75 / 1_000_000 + outputs * 4.50 / 1_000_000
def score_of(data: dict) -> int:
    value = data.get("score")
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise ValueError("평가 응답의 score는 숫자여야 합니다.")
    if not 0 <= value <= 100:
        raise ValueError("평가 응답의 score는 0~100이어야 합니다.")
    return int(value)


## 분류 결과를 검사하는 라우터

In [6]:
# 분류값과 신뢰도를 검사해 사용할 전문 경로를 결정합니다.
ALLOWED = {"billing", "account", "technical"}
def route(decision: dict) -> tuple[str | None, str]:
    category = decision.get("category")
    confidence = float(decision.get("confidence", 0))
    if category not in ALLOWED:
        return None, "허용 목록에 없는 분류"
    if confidence < 0.75:
        return None, "신뢰도 기준 미달"
    return str(category), "전문 경로 사용"
classifier = "billing, account, technical 중 하나로 분류합니다. 혼합 질문은 unknown입니다. category와 confidence를 반환합니다."
valid = call_json(classifier, "카드에서 같은 요금이 두 번 결제됐어요.", 120)
invalid = call_json(classifier, "로그인도 안 되고 카드 청구도 이상해요.", 120)
print("단일 의도:", valid, "->", route(valid))
print("혼합 의도:", invalid, "->", route(invalid))


단일 의도: {'category': 'billing', 'confidence': 0.98} -> ('billing', '전문 경로 사용')
혼합 의도: {'category': 'unknown', 'confidence': 0.93} -> (None, '허용 목록에 없는 분류')


## Before: 분류값을 그대로 사전 키로 사용

In [7]:
# 허용 목록 밖 분류를 사전 키로 쓸 때의 문제를 확인합니다.
prompts = {"billing": "결제 지원 담당자", "account": "계정 지원 담당자",
           "technical": "기술 지원 담당자"}
bad_key = invalid.get("category")
print("분류값:", bad_key)
print("Before 결과:", "KeyError 위험" if bad_key not in prompts else prompts[bad_key])


분류값: unknown
Before 결과: KeyError 위험


## After: 거절하지 않고 필요한 구분을 요청

In [8]:
# 유효한 분류는 전문 경로로 보내고 나머지는 추가 정보를 요청합니다.
def answer_ticket(question: str, decision: dict) -> dict:
    category, reason = route(decision)
    if category is None:
        return {"status": "needs_clarification", "reason": reason,
                "answer": "계정 로그인과 결제 중 먼저 확인할 문제를 골라 주세요."}
    return call_json(prompts[category],
                     f"질문에 2문장으로 답하세요. JSON answer: {question}", 220)
good_answer = answer_ticket("카드에서 같은 요금이 두 번 결제됐어요.", valid)
safe_answer = answer_ticket("로그인도 안 되고 카드 청구도 이상해요.", invalid)
print("전문 경로:", good_answer)
print("안전 처리:", safe_answer)
print("누적 성공 응답:", len(calls), "| 사용량 기준 추정 비용($):", round(cost_usd(calls), 6))


전문 경로: {'answer': '먼저 결제 내역과 주문 상태를 확인해 중복 승인인지 실제 중복 청구인지 구분해 주세요. 같은 요금이 두 번 확정 결제된 경우 카드사와 함께 환불 처리를 진행하겠습니다.'}
안전 처리: {'status': 'needs_clarification', 'reason': '허용 목록에 없는 분류', 'answer': '계정 로그인과 결제 중 먼저 확인할 문제를 골라 주세요.'}
누적 성공 응답: 3 | 사용량 기준 추정 비용($): 0.000603
